<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/03-rag/03-chunking.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 分块

**目标：** 回到流水线的第一个决策——文档怎么切——现在你已经看过检索成功与失败，可以对照真实约束来评判一个分块策略了。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 环境准备

每个 notebook 都是自包含的，接下来的两个单元格会把它从零搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的环境准备 helper，并拉入 `groq` 客户端；本篇还会用到 `requests`。
2. **载入你的 API key。** 免费 key 在 [console.groq.com](https://console.groq.com/) 领（无需信用卡）。在 Colab 里通过左侧边栏的**钥匙图标** → **Add new secret** 添加，名称必须严格写成 `GROQ_API_KEY`，粘贴值，并把 **Notebook access** 打开。本地运行的话，则把 `GROQ_API_KEY` 设成环境变量。

（完整走查和选模型的指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main" requests

In [ ]:
from aien import setup

# 载入 GROQ_API_KEY（Colab Secrets 或本地环境变量）并返回一个可直接
# 使用的 Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用
# 报 404，就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 为什么分块放在最后讲

到目前为止每一篇都在悄悄依赖同一个函数：那个把 RFC 变成 chunk 的 section-aware 切分器。你 embed、检索、融合、重排的，都是它切出来的东西。我们是故意推迟解释它的。分块是数据流经的*第一个*步骤，却是你*最后*才能讲明白的一步，因为一个分块决策只有对照下游的一切才说得通：

- **chunk 大小 vs embedding 模型。** 长过模型输入窗口的 chunk 会被默默截断（notebook 01 里 `all-MiniLM-L6-v2` 的上限是几百个 token）。所以「切多大？」一部分答案由你选的 embedder 给出。
- **chunk 大小 vs 检索精度。** 你在 notebook 01 里看到，一次命中应该主要是*答案*，而不是填充；在 notebook 02 的失效案例里看到，一条边界把答案劈成两个 chunk，是任何 reranker 都修不了的检索 bug。
- **chunk 大小 vs prompt 预算。** 你要把前 k 个 chunk 塞进生成 prompt，所以 chunk 大小 × k 必须塞得下。

现在这三条约束你手上都有了，这正是为什么分块*现在*值得一整篇 notebook，而在第一页讲只会是一堆凭空的数字。（刚进本节？流水线总览见 [00-what-is-rag](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/03-rag/00-what-is-rag.zh.ipynb)。）

## 语料：十篇 IETF RFC

本仓库在整条 RAG 与评测脉络里只用一份语料：十篇 IETF RFC，覆盖 IP、TCP、DNS、HTTP/1.1、BGP、SMTP、WebSocket、OAuth 2.0、HTTP/2 和 HTTP Semantics。notebook 01–02 检索的正是这份语料（它们用了我们马上要拆开的那个分块器的精简复制），第 04 节的评测也是在这些 chunk 上度量检索。

RFC 之所以是好的教学语料，恰恰因为它们的乱法就是真实语料的乱法：每 58 行一个页眉和页脚、页与页之间有换页符、ASCII 图，以及 40 年出版历程里逐渐漂移出来的章节约定。如果你的流水线能干净地处理这些，那么一堆导出的内部 wiki 就吓不到你。


In [ ]:
# 下载共享语料：十篇 IETF RFC，缓存到 data/rfc/。
# 本仓库里每个需要这份语料的 notebook 都包含这个单元格 ——
# 自包含优先于 DRY，好让每个 notebook 都能自己从头跑到尾。
import os
import requests

RFC_NUMBERS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
RFC_TITLES = {
    791: 'IP', 793: 'TCP', 1035: 'DNS', 2616: 'HTTP/1.1', 4271: 'BGP',
    5321: 'SMTP', 6455: 'WebSocket', 6749: 'OAuth 2.0', 7540: 'HTTP/2',
    9110: 'HTTP Semantics',
}
DATA_DIR = 'data/rfc'
os.makedirs(DATA_DIR, exist_ok=True)

corpus = {}  # rfc 编号 -> 原始文本
for num in RFC_NUMBERS:
    path = os.path.join(DATA_DIR, f'rfc{num}.txt')
    if not os.path.exists(path):  # 已缓存：重跑时跳过下载
        resp = requests.get(f'https://www.rfc-editor.org/rfc/rfc{num}.txt', timeout=30)
        resp.raise_for_status()
        with open(path, 'w') as f:
            f.write(resp.text)
    with open(path) as f:
        corpus[num] = f.read()

for num in RFC_NUMBERS:
    print(f'RFC {num:>4}  {RFC_TITLES[num]:<15} {len(corpus[num]):>9,} chars')

## 先清洗，再切分

打开 RFC 2616，你会看到页面杂物：一行像 `Fielding, et al.  Standards Track  [Page 5]` 的页脚、一个换页符，然后是一行像 `RFC 2616  HTTP/1.1  June 1999` 的页眉，每 58 行重复一次。如果放着不管，这些垃圾会落进 chunk 中间，被当成正文一起去 embed。

而且这份语料连「乱」都不统一：RFC 9110（2022）用的是现代格式，完全没有分页；而 1981 年那几篇用的是*另一种*页眉样式（单独一行 `September 1981`），下面的正则抓不到。一份语料里混着几代排版约定，这正是真实文档集合的样子。

下面这个清洗器就是三条正则。它不需要比这更聪明，但它必须存在。跳过这一步是 RAG 流水线里最常见的自伤：chunk 里是垃圾，embedding 里就是垃圾，下游任何检索技巧都修复不了。


In [ ]:
import re

def clean_rfc(text):
    """去掉 RFC 的页面杂物：换页符、页脚、页眉。"""
    text = text.replace('\f', '\n')
    lines = []
    for line in text.split('\n'):
        if re.match(r'^.*\[Page \d+\]\s*$', line):        # 'Author, et al. ... [Page 12]'
            continue
        if re.match(r'^RFC \d+\s+.*\S+ \d{4}\s*$', line):  # 'RFC 9110  HTTP Semantics  June 2022'
            continue
        lines.append(line)
    text = '\n'.join(lines)
    return re.sub(r'\n{3,}', '\n\n', text).strip()  # 把杂物留下的空隙合上

cleaned = {num: clean_rfc(corpus[num]) for num in RFC_NUMBERS}

In [ ]:
# 前后对比：RFC 2616 里同一个分页处，原始文本 vs 清洗后。
raw = corpus[2616]
i = raw.find('[Page 20]')   # 正文中间的一处分页
print('--- RAW (around the page-20 break) ---')
print(repr(raw[i - 150:i + 250]))

# 取页脚之前最后一行真正的内容，然后在清洗后的文本里找到它，
# 这样就能展示同一处区域，且杂物已被去掉。
before = [l for l in raw[:i].split('\n') if len(l.strip()) > 20][-1]
j = cleaned[2616].find(before.strip()[:50])
print()
print('--- CLEANED (same region) ---')
print(repr(cleaned[2616][j:j + 300]))

## 三个分块器，从零写

不用任何库：下面三个策略加起来大约 30 行，自己写一遍能让取舍变得具体。写代码之前先看全局：

| 策略 | 怎么切 | 优点 | 缺点 | 最适用 |
|---|---|---|---|---|
| **固定长度 + 重叠** | 每 `size` 个字符，按 `overlap` 重叠 | 极其简单；chunk 数量和大小可预测 | 无视结构，会在句中、表格中间切开；答案会被切断 | 语料没有可用的结构（聊天记录、OCR 导出） |
| **按段落** | 按空行切，打包到预算上限 | 尊重作者给出的最小单元；句子得以保留 | 标题和正文可能落到不同的 chunk | 有真正段落的散文，标题很少 |
| **按 section** | 按文档标题切，长 section 再按段落打包 | 边界和主题边界一致；标题随正文一起走 | 上限就是标题正则的水准；大小跨度最大 | 语料有真实结构（RFC、Markdown、合同、代码），而大多数都有 |

这三者其实是同一个底层旋钮（*你多大程度尊重文档自身的结构？*），从「完全不尊重」拧到「格式允许多少就尊重多少」。现在看实现：

1. **固定长度加重叠：** 每 `size` 个字符切一刀，按 `overlap` 重叠。它对文本一无所知。重叠是一种对冲：如果一条边界把答案劈成两半，也许相邻的 chunk 还装着完整的那份。
2. **按段落：** 按空行切，再把段落打包成不超过预算的 chunk。它尊重作者给你的最小结构单元。
3. **按 section：** 按 RFC 的章节标题切（`5.3.  Ordering of Field Lines`）。这能奏效是因为 RFC 正文有缩进、而标题顶在零列，所以一条正则就能抓住文档的真实结构。长 section 退化为按段落打包，并把标题一起带上，让每一片都保住自己的上下文。


In [ ]:
def chunk_fixed(text, size=1200, overlap=200):
    chunks, step = [], size - overlap
    for start in range(0, len(text), step):
        piece = text[start:start + size]
        if piece.strip():
            chunks.append(piece)
        if start + size >= len(text):
            break
    return chunks

def chunk_paragraphs(text, max_chars=1200):
    paras = [p for p in text.split('\n\n') if p.strip()]
    chunks, cur = [], ''
    for p in paras:
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur)
            cur = p
        else:
            cur = cur + '\n\n' + p if cur else p
    if cur:
        chunks.append(cur)
    return chunks

HEADING_RE = re.compile(r'^\d+(?:\.\d+)*\.?\s+\S', re.MULTILINE)

def chunk_sections(text, max_chars=2400):
    starts = [m.start() for m in HEADING_RE.finditer(text)]
    if not starts:
        return chunk_paragraphs(text, max_chars)
    chunks = [text[:starts[0]].strip()] if text[:starts[0]].strip() else []
    for a, b in zip(starts, starts[1:] + [len(text)]):
        sec = text[a:b].strip()
        if len(sec) <= max_chars:
            chunks.append(sec)
        else:  # 长 section：按段落打包，并把标题一起带上
            heading = sec.split('\n', 1)[0]
            for sub in chunk_paragraphs(sec, max_chars):
                chunks.append(sub if sub.startswith(heading) else heading + '\n' + sub)
    return [c for c in chunks if c]

## 对比它们：数量与大小分布

chunk 大小是「检索精度」与「上下文成本」之间的权衡。小 chunk 精确（一次命中主要是答案，不是填充），但有把答案切散的风险，而且要在 prompt 里放更多它们才能覆盖一个主题。大 chunk 让答案保持完整，但每次命中都会拖进更多无关文本，既花 token 又稀释模型的注意力。

注意按 section 的分布：它是*最宽*的，因为 section 本来就是长短不一。这不是缺陷。它意味着 chunk 边界落在作者划分主题边界的地方，而不是落在一个字符计数器碰巧停下的地方。


In [ ]:
import statistics

strategies = {
    'fixed (1200/200)': chunk_fixed,
    'paragraph (1200)': chunk_paragraphs,
    'section (2400)':   chunk_sections,
}

all_chunks = {}
for name, fn in strategies.items():
    chunks = []
    for num in RFC_NUMBERS:
        chunks.extend(fn(cleaned[num]))
    all_chunks[name] = chunks

print(f'{"strategy":<18} {"chunks":>7} {"min":>6} {"median":>7} {"max":>7}')
for name, chunks in all_chunks.items():
    sizes = [len(c) for c in chunks]
    print(f'{name:<18} {len(chunks):>7} {min(sizes):>6} '
          f'{int(statistics.median(sizes)):>7} {max(sizes):>7}')

## 那个承重的 demo：一个问题，三种边界行为

拿一个具体的问题：*「当请求体太大时 HTTP 用哪个状态码？」* 答案在 RFC 9110 的 15.5.14 节，**413 Content Too Large**：一个标题后跟两段短段落。

对每个策略，找出提到它的 chunk，看它们从哪开始、到哪结束。这一份打印输出就是关于分块的整套论证。


In [ ]:
PHRASE = 'Content Too Large'   # 状态码 413 在 RFC 9110 里的名字

for name, chunks in all_chunks.items():
    hits = [i for i, c in enumerate(chunks) if PHRASE in c]
    print(f'--- {name}: {len(hits)} chunk(s) mention {PHRASE!r}')
    for i in hits[:3]:
        c = chunks[i]
        print(f'  [chunk {i}, {len(c)} chars]')
        print(f'    starts: {c[:70]!r}')
        print(f'    ends:   {c[-70:]!r}')
    print()

跑一下，注意三件事：

- **固定长度**的、包含该短语的 chunk，通常起止于句子中间，因为字符计数器不在乎 15.5.14 节是否存在。标题 `413 Content Too Large` 和那句解释服务器*何时*发送它的句子，能不能落进同一个 chunk，全靠运气。200 字符的重叠有时能救回来（重叠就是干这个的），但那只是对冲，不是修复：它缩小了切分造成伤害的窗口，但并没有消除它。
- **按段落**的 chunk 结束在段落边界，所以句子得以保留，但标题和它的正文是两个不同段落，打包器可能把它们放进不同 chunk。一个 chunk 说「服务器不愿处理这么大的负载」，而它上面没有 `413` 这个字符串——这种 chunk，关键词搜索永远没法把它和那个问题连起来。
- **按 section** 恰好产出一个 chunk，从 `15.5.14.  413 Content Too Large` 开始，包含整个小节：标题、码、解释一起走。（其他命中只是目录里列了那个短语的行；检索会把真正的这一节排到它们前面，因为它才是在真正讨论这个主题。）

普遍的教训：**当语料有结构时，结构感知胜过聪明的通用做法**，而大多数语料都有结构。RFC 有编号章节；你的文档有 Markdown 标题；合同有条款；代码有函数。先把力气花在找出结构上，再去花在调 chunk 大小上。

有一条要带走的提醒：section 分块器的上限就是它的标题正则。标题不统一的语料（1981 年的 RFC 791 就有一些）会出现误切。信任任何分块器之前，先用眼睛抽查一批 chunk；这个习惯比任何下游修复都便宜。


## 最佳实践与反模式

| ✅ 要做的 | ❌ 反模式 |
|---|---|
| 在切分*之前*清洗页面杂物/套话 | 直接 embed 原始文档，于是 chunk 里是垃圾，向量里也是垃圾 |
| 语料有结构（大多数都有）时用结构感知的切分 | 默认用固定长度，把句子/表格切在中间 |
| 让标题和它的正文待在一起，使 chunk 自包含 | 让标识符和它的解释落到不同 chunk |
| 按 **token** 计量 chunk 大小，对照 embedder 的窗口 | 按字符计量，然后悄悄溢出/被截断 |
| 靠重跑检索、读数字来挑分块器 | 还没法度量检索就凭感觉调 chunk 大小 |
| 信任切分器之前先肉眼抽查一批 chunk | 假设正则在乱糟糟的语料上抓住了每个标题 |

（整套技术栈范围的清单见 [docs/best-practices-and-anti-patterns.md](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/docs/best-practices-and-anti-patterns.md)。）

## 练习

1. **基于 token 的固定分块。** 改写 `chunk_fixed`，让大小按 token 而不是字符来计量（用 `len(text.split())` 作便宜的代理，或上真正的 tokenizer）。重跑对比表。大小分布变紧了吗？为什么生产系统按 token 做预算？（把它串回去：notebook 01 的 embedder 在超出自己的 token 窗口后就会截断，所以真正要紧的正是按 token 计量的大小。）
2. **重叠扫描。** 在 RFC 9110 上用 0、100、200、400 的重叠分别跑 `chunk_fixed`，对每个设置统计：包含 `'413 (Content Too Large)'` 的 chunk 里，有多少也包含短语 `'willing or able to process'`（那一节的正文）。把这个比例画出来或打印出来；你度量的就是这个对冲实际买到了多少。
3. **标题路径元数据。** 扩展 `chunk_sections`，把章节号和标题（例如 `15.5.14. 413 Content Too Large`）作为每个 chunk 的元数据附加上去，而不只是留在文本里。随机打印五个 chunk 及其元数据。这正是 notebook 01 用来做引用的那个 `chunk_meta`，而你现在是在正经地构建它。
4. **重新分块并重跑检索。** 拿 notebook 01 的 embedding 搜索，用 `chunk_fixed` 的输出而不是 `chunk_sections` 重建它的索引，再跑它那三个 demo 查询。从真实结果确认：你选的分块器会推动检索质量——这是整篇 notebook 所依赖的判断。
5. **弄坏 section 分块器。** 在 RFC 791 或 RFC 793（最老的两篇）里找一处 `HEADING_RE` 误切的地方，要么是漏掉的标题，要么是它误报的非标题行。在不破坏 RFC 9110 的前提下修好这条正则，并留意随着语料变老，这件事会变得多麻烦。
